In [ ]:
# import sys
# import subprocess

# path = f'"{sys.executable}"'
# subprocess.check_call(f"{path} -m pip install ollama", shell=True)

# import importlib
# importlib.invalidate_caches()

# import ollama
# print(f"Ollama successfully imported from: {ollama.__file__}")

In [ ]:
import ollama
import random
import torch
import csv
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from eda import clean_data

In [ ]:

# model configs
MODEL_NAME = 'qwen3:14b' 
NUM_SAMPLES_TO_GENERATE = 20000
SIMILARITY_THRESHOLD = 0.85 
N_SHOT = 3
TEMPERATURE = 0.7
OUTPUT_FILE = "synthetic_pcl_data.tsv"

print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
# dataset = "dontpatronizeme_pcl.tsv"
category_dataset = "dontpatronizeme_categories.tsv"

# Load the data, skipping the first 4 rows
category_df = pd.read_csv(category_dataset, sep='\t', skiprows=4, names=['number', 'id', 'text', 'target', 'country', 'start', 'end', 'span', 'category', 'annotators'])
category_df['text'] = category_df['text'].apply(clean_data)
category_df['category'] = category_df['category'].str.strip()
category_df['target'] = category_df['target'].str.strip()

binary_dataset = "dontpatronizeme_pcl.tsv"

binary_df = pd.read_csv(binary_dataset, sep='\t', skiprows=4, names=['number', 'id', 'target', 'country', 'text', 'label'])
binary_df['text'] = binary_df['text'].apply(clean_data)
binary_df["label"] = pd.to_numeric(binary_df["label"])
binary_df['target'] = binary_df['target'].astype(str).str.strip()

# iterating a df
# for index, row in df.iterrows():
#     print(row['text'])

# embed original data so generated data can be compared
original_data = binary_df['text'].dropna().tolist()
print("Embedding dataset...")
original_embeddings = embedder.encode(original_data, convert_to_tensor=True)

binary_df

In [ ]:
category_explanations = {
    "Unbalanced_power_relations": "the 'Saviour' using 'Unbalanced power relations'. The text should subtly position the author in a superior, privileged situation. Distance the author from the community, and express a self-appointed responsibility to grant them rights, help, or opportunities they supposedly cannot achieve themselves (e.g., 'You can make a difference in their lives' or 'They deserve another opportunity').",
    "Shallow_solution": "the 'Saviour' offering a 'Shallow solution'. Present a simple, superficial, or token charitable action by a privileged group as if it is a life-saving miracle or a complete solution to a deep-rooted, systemic problem (e.g., 'Raise money to combat homelessness by sleeping in bags for one night' or 'Donating one box makes a real difference').",
    "Presupposition": "the 'Expert' using 'Presupposition'. Assume things about the community's situation without evidence, relying heavily on stereotypes or clichés. Present subjective assumptions about what they can or cannot do as absolute, unquestionable facts (e.g., 'elderly people who are simply unable to evacuate due to physical limitations').",
    "Authority_voice": "the 'Expert' using an 'Authority voice'. The text should sound like the author has appointed themselves as the spokesperson for the vulnerable group. Unsolicitedly explain their own situation to them, or advise them on how they should feel or act to overcome their struggles, completely ignoring their own agency (e.g., 'Accepting their situation is the first step to having a normal life').",
    "Metaphors": "the 'Poet' using 'Metaphor'. Use euphemisms, literary comparisons, and poetic metaphors to soften or romanticize a harsh reality. Conceal the condescension behind flowery language that depicts their struggles in a cinematic or literary light (e.g., 'those who cling to boats to reach a shore of survival').",
    "Compassion": "the 'Poet' using weaponized 'Compassion'. Focus heavily on how needy and pitiful the community is to evoke sorrow and guilt from the audience. Use overly dramatic, flowery descriptions of their vulnerability and suffering rather than providing objective, factual information (e.g., 'discarded in the streets, resigned to selling fake bags').",
    "The_poorer_the_merrier": "the 'Poet' claiming 'The poorer, the merrier'. Romanticize their vulnerability by claiming it makes them better, stronger, or happier people. Push the narrative that living in poverty or disability builds admirable resilience or teaches the privileged class a lesson about the true meaning of life (e.g., 'poor people are happier because they lack material goods' or 'they remind us of the true meaning of hope')."
}
target_explanations = {
    "disabled": "individuals with physical, cognitive, or developmental disabilities. Focus on their daily lives or achievements.",
    "homeless": "people experiencing homelessness, living on the streets, or residing in temporary shelters.",
    "hopeless": "individuals trapped in desperate socio-economic situations, extreme cyclical poverty, or inescapable hardship.",
    "immigrant": "immigrants who have relocated to a new country and are navigating systemic barriers, xenophobia, or cultural integration.",
    "in-need": "communities or individuals requiring urgent humanitarian, financial, or social assistance to meet basic survival needs.",
    "migrant": "migrant workers or displaced individuals on the move for economic survival, often lacking legal protections or permanent status.",
    "poor-families": "low-income families struggling to afford basic necessities like food, reliable housing, or education for their children.",
    "refugee": "refugees or asylum seekers who were forced to flee their home countries due to war, persecution, or natural disaster.",
    "vulnerable": "marginalized populations who are highly susceptible to systemic inequality, exploitation, or crisis due to a lack of societal power.",
    "women": "women, specifically in contexts where patriarchal norms might subtly undermine their agency, independence, or professional capabilities."
}
label_explanations = {
    0: "objective, respectful, and neutral text about vulnerable groups. The text must be purely factual reporting or genuine advocacy without any patronizing tone, savior complex, or pity.",
    1: "realistic Patronizing and Condescending Language (PCL). Mimic the subtle, implicit condescension seen in the examples."
}

positive_cliches = ['poor', 'helpless', 'save', 'souls', 'brave', 'inspiring', 'individuals', 'transient']

In [ ]:

def get_filtered_df(category, target, bin_label):
    if bin_label == 1:
        x = category_df[(category_df['category'] == category) & (category_df['target'] == target)]['text']
    else:
        x = binary_df[(binary_df['target'] == target) & (binary_df['label'] <= 1)]['text']
    if len(x) == 0:
        print(f"failed with c({category}) t({target}) l({bin_label})")
        with open("errors.txt", 'a') as f:
            f.write(f"failed with c({category}) t({target}) l({bin_label})")
    else:
        print(f"succeeded {len(x)} with c({category}) t({target}) l({bin_label})")
    return x
        

def generate_sample(filtered_df, category, target, bin_label, length):
    sampled_examples = filtered_df.sample(n=N_SHOT)
    banned_words = random.sample(positive_cliches, 3)

    system_prompt = f"""You are an NLP researcher creating synthetic data for a sociological study.
    Your task is to write a sentence that perfectly mimics {label_explanations[bin_label]}.
    The vulnerable group the sentence should target is: {target_explanations[target]}.
    {f"The specific style of condescension the author should use is: {category_explanations[category]}." if bin_label == "PCL" else ""}

    CRITICAL INSTRUCTIONS:
    1. DO NOT use overt hate speech or slurs.
    2. DO NOT rely on the exact target keyword. Describe their situation naturally.
    3. DO NOT use these cliché words: {', '.join(banned_words)}.
    4. Output ONLY the generated text.
    5. Aim for around {length} words."""

    examples_text = "\n".join([f"- {ex}" for ex in sampled_examples])
    
    try:
        response = ollama.chat(model=MODEL_NAME, messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': f"Here are {N_SHOT} examples. Generate ONE new, highly nuanced example that is completely different. Return ONLY the sentence:\n{examples_text}"}
        ], options={'temperature': TEMPERATURE})
        return response['message']['content'].strip()
    except Exception as e:
        print(f"Generation error: {e}")
        with open("errors.txt", 'a') as f:
            f.write(f"Generation error: {e}")
        return None

# uncomment to create/overwrite new file
# with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
#     writer = csv.writer(f, delimiter='\t')

generated_count = 0

print("Starting generation loop...")
while generated_count < NUM_SAMPLES_TO_GENERATE:
    bin_label = random.choice([0, 1])
    category = random.choice(list(category_explanations.keys()))
    target = random.choice(list(target_explanations.keys()))
    length = round(random.normalvariate(51, 20), -1)  # round to nearest 10
    examples = get_filtered_df(category, target, bin_label)
    if len(examples) == 0:
        continue
    candidate_text = generate_sample(examples, category, target, bin_label, length)
    
    if not candidate_text:
        continue

    # Semantic Deduplication
    candidate_embedding = embedder.encode(candidate_text, convert_to_tensor=True)
    cosine_scores = util.cos_sim(candidate_embedding, original_embeddings)[0]
    max_similarity = torch.max(cosine_scores).item()
    
    if max_similarity >= SIMILARITY_THRESHOLD:
        print(f"REJECTED (Label {bin_label}, Category {category}, Target {target}, Score {max_similarity:.2f}): {candidate_text}")
        with open("errors.txt", 'a') as f:
            f.write(f"REJECTED (Label {bin_label}, Category {category}, Target {target}, Score {max_similarity:.2f}): {candidate_text}")
    else:
        print(f"ACCEPTED (Label {bin_label}, Category {category}, Target {target}, Score {max_similarity:.2f}): {candidate_text}")
        
        # Append immediately to the TSV file
        with open(OUTPUT_FILE, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f, delimiter='\t')
            # Columns: 0='X', 1='X', 2='X', 3='X', 4=Text, 5=Label
            writer.writerow(['X', 'X', 'X', 'X', candidate_text, bin_label * 2])  # x2 so that label 1 -> score 2 = PCL
            
        generated_count += 1
        
        # Add the new embedding to our reference tensor
        original_embeddings = torch.cat((original_embeddings, candidate_embedding.unsqueeze(0)), 0)

print(f"Success! {NUM_SAMPLES_TO_GENERATE} rows saved to {OUTPUT_FILE}")